# GLYPH8-R32 + MWA-v2 V21 ChunkTrainLock

Fresh LoRA-only Kaggle submission notebook. Produces `/kaggle/working/submission.zip` containing exactly `adapter_config.json` and `adapter_model.safetensors`. Chunked training is enabled by default to reduce OOM risk.

In [ ]:

# =============================================================================
# GLYPH8-R32 + MWA-v2 FRESH LoRA-ONLY NEMOTRON SUBMISSION NOTEBOOK - V21 CHUNKTRAINLOCK
# NVIDIA Nemotron Model Reasoning Challenge
#
# Produces exactly: /kaggle/working/submission.zip
# Zip root contains exactly:
#   adapter_config.json
#   adapter_model.safetensors
#
# Hard rules:
# - No existing-adapter fallback.
# - No leaderboard adapter copying.
# - No dummy/empty safetensors.
# - No base model/tokenizer files in submission.zip.
# - LoRA rank <= 32.
# - Map before training: category map + optional base-loss probe.
# - Train fresh LoRA adapter from the pinned metric base model.
# =============================================================================

from __future__ import annotations

import os
import re
import gc
import csv
import json
import math
import time
import glob
import zipfile
import random
import shutil
import hashlib
import unicodedata
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

warnings.filterwarnings('ignore')

# Must be set before torch CUDA allocator is initialized.
# v18 memory lock: must be set before torch/transformers touch CUDA.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = os.environ.get('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128')
os.environ.setdefault('CUDA_MODULE_LOADING', 'LAZY')
os.environ.setdefault('HF_ENABLE_PARALLEL_LOADING', 'false')
os.environ.setdefault('HF_PARALLEL_LOADING_WORKERS', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('TRANSFORMERS_NO_ADVISORY_WARNINGS', '1')
os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')
os.environ.setdefault('SAFETENSORS_FAST_GPU', '0')

MAX_LORA_RANK = 32
PINNED_BASE_MODEL_PATH = '/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1'

# -----------------------------
# 0. Configuration
# -----------------------------
@dataclass
class CFG:
    input_root: str = os.environ.get('INPUT_ROOT', '/kaggle/input')
    work_dir: str = os.environ.get('WORK_DIR', '/kaggle/working/fresh_mapped_lora')
    adapter_dir: str = os.environ.get('ADAPTER_DIR', '/kaggle/working/fresh_mapped_lora_adapter')
    output_zip: str = os.environ.get('OUTPUT_ZIP', '/kaggle/working/submission.zip')

    # Pinned model path supplied by user. Override only if intentionally testing.
    base_model_name: str = os.environ.get('BASE_MODEL_PATH', PINNED_BASE_MODEL_PATH)
    public_base_model_name: str = os.environ.get('PUBLIC_BASE_MODEL_NAME', os.environ.get('BASE_MODEL_PATH', PINNED_BASE_MODEL_PATH))

    # Modes: train | map_only | selftest
    # There is deliberately no package_existing mode.
    run_mode: str = os.environ.get('RUN_MODE', 'train')

    # LoRA / QLoRA
    lora_rank: int = int(os.environ.get('LORA_RANK', '32'))
    lora_alpha: int = int(os.environ.get('LORA_ALPHA', '64'))
    lora_dropout: float = float(os.environ.get('LORA_DROPOUT', '0.0'))
    learning_rate: float = float(os.environ.get('LR', '5e-5'))
    max_seq_len: int = int(os.environ.get('MAX_SEQ_LEN', '256'))
    train_batch_size: int = int(os.environ.get('TRAIN_BATCH_SIZE', '1'))
    grad_accum_steps: int = int(os.environ.get('GRAD_ACCUM_STEPS', '16'))
    num_epochs: float = float(os.environ.get('NUM_EPOCHS', '1'))
    max_steps: int = int(os.environ.get('MAX_STEPS', '40'))
    warmup_ratio: float = float(os.environ.get('WARMUP_RATIO', '0.03'))
    weight_decay: float = float(os.environ.get('WEIGHT_DECAY', '0.01'))
    max_grad_norm: float = float(os.environ.get('MAX_GRAD_NORM', '1.0'))
    bf16: int = int(os.environ.get('BF16', '0'))  # T4 prior-fix: use FP16, not BF16
    qlora_4bit: int = int(os.environ.get('QLORA_4BIT', '1'))
    bitsandbytes_min_version: str = os.environ.get('BITSANDBYTES_MIN_VERSION', '0.46.1')
    gradient_checkpointing: int = int(os.environ.get('GRADIENT_CHECKPOINTING', '1'))
    force_device_map: str = os.environ.get('FORCE_DEVICE_MAP', 'auto').strip()  # prior-fix default: auto on T4 x2
    gpu_headroom_gib: float = float(os.environ.get('GPU_HEADROOM_GIB', '6.0'))
    single_gpu_max_memory_gib: float = float(os.environ.get('SINGLE_GPU_MAX_MEMORY_GIB', '8.0'))
    cpu_offload_memory: str = os.environ.get('CPU_OFFLOAD_MEMORY', '96GiB')
    offload_folder: str = os.environ.get('OFFLOAD_FOLDER', '/kaggle/working/model_offload')
    # V21: force deep CPU/disk offload during model materialization; add Glyph8-R32 and chunked training.
    gpu0_load_cap_gib: float = float(os.environ.get('GPU0_LOAD_CAP_GIB', '6.0'))
    gpu1_load_cap_gib: float = float(os.environ.get('GPU1_LOAD_CAP_GIB', '12.0'))
    cpu_load_cap: str = os.environ.get('CPU_LOAD_CAP', '120GiB')
    strict_disk_offload: int = int(os.environ.get('STRICT_DISK_OFFLOAD', '1'))
    require_two_gpus_for_30b: int = int(os.environ.get('REQUIRE_TWO_GPUS_FOR_30B', '0'))
    lora_target_profile: str = os.environ.get('LORA_TARGET_PROFILE', 'minimal').strip().lower()  # score | memory | v18 | minimal
    lora_target_modules: str = os.environ.get('LORA_TARGET_MODULES', 'q_proj,v_proj')
    # Model Weight Atlas Lite: metadata/coordinate map only. No full W clone/SVD/residual in submission.
    mwa_enabled: int = int(os.environ.get('MWA_ENABLED', '1'))
    mwa_tile_count: int = int(os.environ.get('MWA_TILE_COUNT', '16'))
    mwa_output_json: str = os.environ.get('MWA_OUTPUT_JSON', '/kaggle/working/fresh_mapped_lora/model_weight_atlas_lite.json')

    # Data discovery / mapping
    max_real_rows: int = int(os.environ.get('MAX_REAL_ROWS', '3000'))
    min_train_rows: int = int(os.environ.get('MIN_TRAIN_ROWS', '1'))
    dev_fraction: float = float(os.environ.get('DEV_FRACTION', '0.0'))
    probe_base_loss: int = int(os.environ.get('PROBE_BASE_LOSS', '0'))
    probe_max_rows: int = int(os.environ.get('PROBE_MAX_ROWS', '8'))
    seed: int = int(os.environ.get('SEED', '918'))

    # Token-level score-aware loss weights.
    reasoning_loss_weight: float = float(os.environ.get('REASONING_LOSS_WEIGHT', '0.35'))
    final_line_loss_weight: float = float(os.environ.get('FINAL_LINE_LOSS_WEIGHT', '2.5'))
    boxed_answer_loss_weight: float = float(os.environ.get('BOXED_ANSWER_LOSS_WEIGHT', '4.0'))

    # Fail-safe packaging policy.
    exact_zip_two_files: int = int(os.environ.get('EXACT_ZIP_TWO_FILES', '1'))
    min_adapter_bytes: int = int(os.environ.get('MIN_ADAPTER_BYTES', '500000'))

    # GlyphMatics Glyph8 transport curriculum. Codec is lossless; LoRA learns the protocol.
    glyph8_enabled: int = int(os.environ.get('GLYPH8_ENABLED', '1'))
    glyph8_max_records: int = int(os.environ.get('GLYPH8_MAX_RECORDS', '384'))
    glyph8_max_payload_bytes: int = int(os.environ.get('GLYPH8_MAX_PAYLOAD_BYTES', '96'))
    glyph8_encode_ratio: float = float(os.environ.get('GLYPH8_ENCODE_RATIO', '0.34'))
    glyph8_decode_ratio: float = float(os.environ.get('GLYPH8_DECODE_RATIO', '0.33'))
    glyph8_repair_ratio: float = float(os.environ.get('GLYPH8_REPAIR_RATIO', '0.33'))
    glyph8_loss_multiplier: float = float(os.environ.get('GLYPH8_LOSS_MULTIPLIER', '1.18'))
    glyph8_manifest_file: str = os.environ.get('GLYPH8_MANIFEST_FILE', '/kaggle/working/fresh_mapped_lora/glyph8_transport_manifest.json')


    # ChunkTrainLock: train in small slices to avoid activation/optimizer memory spikes.
    chunked_training: int = int(os.environ.get('CHUNKED_TRAINING', '1'))
    train_chunk_size: int = int(os.environ.get('TRAIN_CHUNK_SIZE', '192'))
    chunk_max_steps: int = int(os.environ.get('CHUNK_MAX_STEPS', '4'))
    max_train_chunks: int = int(os.environ.get('MAX_TRAIN_CHUNKS', '0'))  # 0 = derive from MAX_STEPS
    save_after_each_chunk: int = int(os.environ.get('SAVE_AFTER_EACH_CHUNK', '1'))
    clear_cache_each_chunk: int = int(os.environ.get('CLEAR_CACHE_EACH_CHUNK', '1'))

cfg = CFG()
Path(cfg.work_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.adapter_dir).parent.mkdir(parents=True, exist_ok=True)
random.seed(cfg.seed)

class Fatal(RuntimeError):
    pass

def log(msg: str) -> None:
    print(f'[fresh-lora] {msg}', flush=True)

def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf-8')

def sha256_file(path: Path, block: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        while True:
            b = f.read(block)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

log('Configuration:')
print(json.dumps(asdict(cfg), indent=2), flush=True)

if cfg.lora_rank > MAX_LORA_RANK:
    raise Fatal(f'LORA_RANK={cfg.lora_rank} exceeds competition max rank {MAX_LORA_RANK}')
if cfg.run_mode not in {'train', 'map_only', 'selftest'}:
    raise Fatal(f'Unsupported RUN_MODE={cfg.run_mode}; allowed: train | map_only | selftest')

# -----------------------------
# 1. Text cleanup, answer extraction, category inference
# -----------------------------
ZERO_WIDTH = ['\u200B', '\u200C', '\u200D', '\uFEFF', '\u202A', '\u202B', '\u202C', '\u202D', '\u202E']
HOMOGLYPHS = {'\u0430': 'a', '\u0435': 'e', '\u043e': 'o', '\u0440': 'p', '\u0441': 'c', '\u0445': 'x', '\u0456': 'i', '\u03bf': 'o'}

PROMPT_COLS = ['prompt', 'problem', 'question', 'input', 'text', 'statement', 'task']
ANSWER_COLS = ['answer', 'target', 'label', 'output', 'solution', 'final_answer', 'expected']
RATIONALE_COLS = ['rationale', 'reasoning', 'cot', 'explanation', 'solution_text', 'work']
CATEGORY_COLS = ['category', 'type', 'task_type', 'problem_type', 'family']
ID_COLS = ['id', 'row_id', 'problem_id', 'sample_id']

CATEGORY_WEIGHTS = {
    'cipher': 1.38,
    'substitution_cipher': 1.38,
    'cryptarithm': 1.42,
    'bit_manipulation': 1.45,
    'equation_numeric': 1.45,
    'numeral': 1.12,
    'unit_conversion': 1.05,
    'gravity': 1.05,
    'physics': 1.05,
    'trajectory': 1.20,
    'logic': 1.18,
    'unknown': 1.25,
}

CATEGORY_ORDER = [
    'cipher', 'substitution_cipher', 'numeral', 'unit_conversion', 'gravity', 'physics',
    'bit_manipulation', 'equation_numeric', 'cryptarithm', 'trajectory', 'logic', 'unknown'
]
CATEGORY_RANK = {c: i for i, c in enumerate(CATEGORY_ORDER)}

def sanitize_text(x: Any, max_len: int = 24000) -> str:
    s = '' if x is None else str(x)
    s = unicodedata.normalize('NFKC', s)
    for z in ZERO_WIDTH:
        s = s.replace(z, '')
    for h, c in HOMOGLYPHS.items():
        s = s.replace(h, c)
    s = s.replace('\x00', '')
    s = s.replace('\r\n', '\n').replace('\r', '\n')
    s = re.sub(r'[ \t]+', ' ', s)
    s = re.sub(r'\n{4,}', '\n\n\n', s).strip()
    return s[:max_len]

def extract_boxed(text: Any) -> Optional[str]:
    s = '' if text is None else str(text)
    matches = re.findall(r'\\boxed\s*\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}', s, flags=re.S)
    if matches:
        return matches[-1].strip()
    return None

def normalize_answer(ans: Any) -> str:
    if ans is None:
        return ''
    s = str(ans).strip()
    boxed = extract_boxed(s)
    if boxed is not None:
        s = boxed
    s = s.strip().strip('`$')
    s = s.replace('\\text{', '').replace('}', '')
    s = re.sub(r'\s+', ' ', s).strip()
    try:
        if re.fullmatch(r'[-+]?\d+\.0+', s):
            return str(int(float(s)))
        if re.fullmatch(r'[-+]?\d*\.\d+(?:[eE][-+]?\d+)?', s):
            f = float(s)
            if math.isfinite(f):
                if abs(f - round(f)) < 1e-12:
                    return str(int(round(f)))
                return f'{f:.12g}'.replace('+', '')
    except Exception:
        pass
    return s

def boxed(ans: Any) -> str:
    return '\\boxed{' + normalize_answer(ans) + '}'

def infer_category(prompt: Any) -> str:
    p = sanitize_text(prompt).lower()
    if any(w in p for w in ['cryptarithm', 'letter is a digit', 'each letter', 'send + more', 'alphametic']):
        return 'cryptarithm'
    if any(w in p for w in ['substitution cipher', 'caesar', 'vigenere', 'ciphertext', 'plaintext', 'decrypt', 'encrypt']):
        return 'substitution_cipher'
    if any(w in p for w in ['cipher', 'decode', 'encoded message']):
        return 'cipher'
    if any(w in p for w in ['xor', 'bit', 'mask', 'rotate', 'shift', 'binary operator', 'modulo 2', 'and/or']):
        return 'bit_manipulation'
    if any(w in p for w in ['equation', 'solve for', 'root of', 'quadratic', 'linear system', 'polynomial']):
        return 'equation_numeric'
    if any(w in p for w in ['base-', 'base ', 'binary', 'octal', 'hexadecimal', 'decimal representation', 'convert from base']):
        return 'numeral'
    if any(w in p for w in ['unit conversion', 'meters', 'kilometers', 'seconds', 'hours', 'grams', 'kg', 'm/s']):
        return 'unit_conversion'
    if any(w in p for w in ['gravity', 'gravitational', 'newton', 'orbit']):
        return 'gravity'
    if any(w in p for w in ['physics', 'velocity', 'acceleration', 'force', 'energy', 'mass']):
        return 'physics'
    if any(w in p for w in ['trajectory', 'projectile', 'path']):
        return 'trajectory'
    if any(w in p for w in ['logic', 'truth', 'proposition', 'if and only if', 'knights and knaves']):
        return 'logic'
    return 'unknown'

def first_present(d: Dict[str, Any], names: Sequence[str]) -> Optional[Any]:
    low = {str(k).lower(): k for k in d.keys()}
    for n in names:
        if n in d and d[n] not in [None, '']:
            return d[n]
        lk = low.get(n.lower())
        if lk is not None and d[lk] not in [None, '']:
            return d[lk]
    return None

# -----------------------------
# 2. Real training-row discovery
# -----------------------------
def looks_like_model_or_adapter_path(path: Path) -> bool:
    parts = [p.lower() for p in path.parts]
    bad = {'transformers', 'default', 'models', 'adapter_model.safetensors'}
    if 'adapter_model.safetensors' in path.name.lower():
        return True
    if any(p in {'pytorch_model.bin', 'model.safetensors'} for p in parts):
        return True
    # Do not exclude every /models/ path because data may live under arbitrary dataset names.
    # Exclude if the file is inside a directory that already contains model config/shards.
    for parent in [path.parent, path.parent.parent if path.parent.parent else path.parent]:
        if (parent / 'config.json').exists() and (list(parent.glob('*.safetensors')) or list(parent.glob('*.bin'))):
            return True
    return False

def iter_candidate_files(root: str) -> List[Path]:
    exts = ['*.jsonl', '*.json', '*.csv', '*.tsv', '*.parquet']
    files = []
    for ext in exts:
        files.extend(Path(root).rglob(ext))
    out = []
    for p in sorted(set(files)):
        if not p.is_file():
            continue
        if looks_like_model_or_adapter_path(p):
            continue
        name = p.name.lower()
        if name in {'adapter_config.json', 'config.json', 'generation_config.json', 'tokenizer.json', 'tokenizer_config.json'}:
            continue
        try:
            if p.stat().st_size == 0 or p.stat().st_size > 2_000_000_000:
                continue
        except Exception:
            continue
        out.append(p)
    return out

def flatten_json_obj(obj: Any) -> Iterable[Dict[str, Any]]:
    if isinstance(obj, dict):
        # Common wrappers.
        for key in ['data', 'train', 'rows', 'examples', 'records', 'items']:
            v = obj.get(key)
            if isinstance(v, list):
                for x in v:
                    if isinstance(x, dict):
                        yield x
                return
        # Single row dict.
        yield obj
    elif isinstance(obj, list):
        for x in obj:
            if isinstance(x, dict):
                yield x

def load_records_from_file(path: Path) -> Iterable[Dict[str, Any]]:
    suffix = path.suffix.lower()
    try:
        if suffix == '.jsonl':
            with path.open('r', encoding='utf-8', errors='ignore') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        obj = json.loads(line)
                    except Exception:
                        continue
                    if isinstance(obj, dict):
                        yield obj
        elif suffix == '.json':
            text = path.read_text(encoding='utf-8', errors='ignore')
            try:
                obj = json.loads(text)
            except Exception:
                return
            yield from flatten_json_obj(obj)
        elif suffix in {'.csv', '.tsv'}:
            dialect = 'excel-tab' if suffix == '.tsv' else 'excel'
            with path.open('r', encoding='utf-8', errors='ignore', newline='') as f:
                reader = csv.DictReader(f, dialect=dialect)
                for row in reader:
                    yield dict(row)
        elif suffix == '.parquet':
            try:
                import pandas as pd
                df = pd.read_parquet(path)
                for row in df.to_dict(orient='records'):
                    yield row
            except Exception:
                return
    except Exception as e:
        log(f'skip unreadable {path}: {e}')
        return

def record_to_training_row(d: Dict[str, Any], source: Path, idx: int) -> Optional[Dict[str, Any]]:
    prompt = first_present(d, PROMPT_COLS)
    answer = first_present(d, ANSWER_COLS)
    if prompt is None or answer is None:
        return None
    prompt_s = sanitize_text(prompt)
    ans_s = normalize_answer(answer)
    if len(prompt_s) < 3 or len(ans_s) < 1:
        return None
    rationale = first_present(d, RATIONALE_COLS)
    rationale_s = sanitize_text(rationale) if rationale is not None else ''
    # Remove stale boxed lines from rationale so the final line is canonical.
    rationale_s = re.sub(r'The answer is\s*\\boxed\s*\{.*?\}\.?', '', rationale_s, flags=re.I | re.S).strip()
    category = first_present(d, CATEGORY_COLS)
    category_s = sanitize_text(category).lower().replace(' ', '_') if category else infer_category(prompt_s)
    if category_s not in CATEGORY_WEIGHTS:
        category_s = infer_category(prompt_s)
    rid = first_present(d, ID_COLS) or f'{source.name}:{idx}'
    return {
        'id': sanitize_text(rid, 256),
        'prompt': prompt_s,
        'answer': ans_s,
        'rationale': rationale_s,
        'category': category_s,
        'source': str(source),
        'source_index': idx,
        'base_loss': None,
        'score_weight': float(CATEGORY_WEIGHTS.get(category_s, CATEGORY_WEIGHTS['unknown'])),
    }

def discover_real_training_rows() -> List[Dict[str, Any]]:
    files = iter_candidate_files(cfg.input_root)
    log(f'candidate data files: {len(files)}')
    rows = []
    seen = set()
    for path in files:
        if len(rows) >= cfg.max_real_rows:
            break
        n0 = len(rows)
        for i, rec in enumerate(load_records_from_file(path)):
            row = record_to_training_row(rec, path, i)
            if row is None:
                continue
            key = hashlib.sha256((row['prompt'] + '\n' + row['answer']).encode('utf-8', errors='ignore')).hexdigest()
            if key in seen:
                continue
            seen.add(key)
            rows.append(row)
            if len(rows) >= cfg.max_real_rows:
                break
        if len(rows) > n0:
            log(f'loaded {len(rows)-n0} rows from {path}')
    if len(rows) < cfg.min_train_rows:
        raise Fatal(f'No valid real training rows found under {cfg.input_root}. Refusing to create fake/empty adapter.')
    # Curriculum order: stable/known families first, weak/mutable families later.
    rows.sort(key=lambda r: (CATEGORY_RANK.get(r['category'], 999), len(r['prompt'])))
    return rows

def summarize_rows(rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    counts: Dict[str, int] = {}
    sources: Dict[str, int] = {}
    for r in rows:
        counts[r['category']] = counts.get(r['category'], 0) + 1
        sources[r['source']] = sources.get(r['source'], 0) + 1
    lane_counts: Dict[str, int] = {}
    for r in rows:
        lane_counts[str(r.get('lane', 'reason'))] = lane_counts.get(str(r.get('lane', 'reason')), 0) + 1
    summary = {
        'total_rows': len(rows),
        'lane_counts': dict(sorted(lane_counts.items(), key=lambda kv: (-kv[1], kv[0]))),
        'category_counts': dict(sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))),
        'top_sources': dict(sorted(sources.items(), key=lambda kv: -kv[1])[:25]),
        'category_weights': CATEGORY_WEIGHTS,
        'curriculum_order': CATEGORY_ORDER,
    }
    write_json(Path(cfg.work_dir) / 'mapping_summary_pre_probe.json', summary)
    return summary


# -----------------------------
# 3. GlyphMatics GLYPH8 codec + transport curriculum
# -----------------------------
BRAILLE_BASE = 0x2800
GLYPH8_OPEN = '⟦GLYPH8'
GLYPH8_CLOSE = '⟦/GLYPH8⟧'

def glyph8_encode_bytes(data: bytes) -> str:
    """Lossless byte -> 8-dot Braille glyph string. 1 byte = 1 Unicode Braille cell."""
    return ''.join(chr(BRAILLE_BASE + b) for b in data)

def glyph8_decode_bytes(glyphs: str) -> bytes:
    out = bytearray()
    for ch in glyphs:
        cp = ord(ch)
        if BRAILLE_BASE <= cp <= BRAILLE_BASE + 255:
            out.append(cp - BRAILLE_BASE)
    return bytes(out)

def glyph8_make_frame(text: str, domain: str = 'unknown', task_type: str = 'payload', max_bytes: Optional[int] = None) -> Dict[str, str]:
    raw = sanitize_text(text).encode('utf-8', errors='strict')
    truncated = False
    if max_bytes is not None and len(raw) > max_bytes:
        raw = raw[:max_bytes]
        # Avoid invalid UTF-8 tail in preview text; raw bytes remain exact for payload.
        truncated = True
    payload = glyph8_encode_bytes(raw)
    sha = hashlib.sha256(raw).hexdigest()
    header = f'{GLYPH8_OPEN}|v=1|len={len(raw)}|sha256={sha}|domain={domain}|task={task_type}|truncated={int(truncated)}⟧'
    frame = header + '\n' + payload + '\n' + GLYPH8_CLOSE
    preview = raw.decode('utf-8', errors='replace')
    return {'frame': frame, 'payload': payload, 'sha256': sha, 'length': str(len(raw)), 'text': preview, 'truncated': str(int(truncated))}

def glyph8_verify_frame(frame: str) -> Tuple[bool, Dict[str, Any]]:
    lines = [ln for ln in str(frame).splitlines() if ln.strip()]
    if len(lines) < 3 or not lines[0].startswith(GLYPH8_OPEN) or lines[-1].strip() != GLYPH8_CLOSE:
        return False, {'error': 'bad_frame_markers'}
    header = lines[0]
    payload = ''.join(lines[1:-1])
    meta: Dict[str, Any] = {}
    for part in header.strip('⟦⟧').split('|')[1:]:
        if '=' in part:
            k, v = part.split('=', 1)
            meta[k] = v
    raw = glyph8_decode_bytes(payload)
    meta['decoded_len'] = len(raw)
    meta['decoded_sha256'] = hashlib.sha256(raw).hexdigest()
    ok = str(meta.get('len')) == str(len(raw)) and str(meta.get('sha256')) == meta['decoded_sha256']
    return bool(ok), meta

def glyph8_corrupt_frame(frame: str) -> str:
    """Deterministic tiny corruption for repair-lane examples."""
    chars = list(frame)
    for i, ch in enumerate(chars):
        cp = ord(ch)
        if BRAILLE_BASE <= cp <= BRAILLE_BASE + 255:
            chars[i] = chr(BRAILLE_BASE + ((cp - BRAILLE_BASE + 1) % 256))
            break
    return ''.join(chars)

def glyph8_record_id(base: Dict[str, Any], lane: str) -> str:
    return f"{base.get('id','row')}::glyph8::{lane}"

def make_glyph8_records(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Create verified encode/decode/repair traces. These rows teach protocol behavior, not content storage."""
    if not cfg.glyph8_enabled or cfg.glyph8_max_records <= 0:
        return []
    out: List[Dict[str, Any]] = []
    source_rows = rows[:max(1, min(len(rows), cfg.glyph8_max_records))]
    for i, r in enumerate(source_rows):
        cat = r.get('category', 'unknown')
        # Use compact payloads so the protocol fits into small T4-safe max_seq_len.
        payload_text = r.get('prompt', '')
        fr = glyph8_make_frame(payload_text, domain=cat, task_type='problem', max_bytes=cfg.glyph8_max_payload_bytes)
        ok, meta = glyph8_verify_frame(fr['frame'])
        if not ok:
            continue
        lane_mod = i % 3
        base = dict(r)
        if lane_mod == 0:
            row = {
                **base,
                'id': glyph8_record_id(base, 'encode'),
                'lane': 'glyph8_encode',
                'prompt': 'Encode this UTF-8 text into a GLYPH8 Braille byte frame with manifest and checksum. Text:\n' + fr['text'],
                'rationale': 'Verified byte-exact GLYPH8 frame:\n' + fr['frame'],
                'answer': fr['sha256'][:16],
                'category': cat,
                'score_weight': float(base.get('score_weight', 1.0)) * cfg.glyph8_loss_multiplier,
            }
        elif lane_mod == 1:
            row = {
                **base,
                'id': glyph8_record_id(base, 'decode'),
                'lane': 'glyph8_decode',
                'prompt': 'Decode this GLYPH8 Braille byte frame and verify its SHA-256 manifest. Frame:\n' + fr['frame'],
                'rationale': 'Decoded UTF-8 text:\n' + fr['text'] + f"\nVerifier: len={meta.get('decoded_len')} sha256={meta.get('decoded_sha256')}",
                'answer': fr['sha256'][:16],
                'category': cat,
                'score_weight': float(base.get('score_weight', 1.0)) * cfg.glyph8_loss_multiplier,
            }
        else:
            bad = glyph8_corrupt_frame(fr['frame'])
            row = {
                **base,
                'id': glyph8_record_id(base, 'repair'),
                'lane': 'glyph8_repair',
                'prompt': 'Repair this corrupted GLYPH8 frame so the decoded bytes match the manifest. Corrupted frame:\n' + bad,
                'rationale': 'Corrected verified GLYPH8 frame:\n' + fr['frame'],
                'answer': fr['sha256'][:16],
                'category': cat,
                'score_weight': float(base.get('score_weight', 1.0)) * cfg.glyph8_loss_multiplier,
            }
        out.append(row)
    manifest = {
        'name': 'GLYPH8-R32 transport curriculum',
        'lossless_boundary': 'UTF-8 bytes <-> U+2800..U+28FF codec + checksum verifier; LoRA only learns protocol behavior.',
        'records': len(out),
        'max_payload_bytes': cfg.glyph8_max_payload_bytes,
        'lanes': summarize_lanes(out),
        'submission_includes_manifest': False,
    }
    write_json(Path(cfg.glyph8_manifest_file), manifest)
    return out

def summarize_lanes(rows: List[Dict[str, Any]]) -> Dict[str, int]:
    d: Dict[str, int] = {}
    for r in rows:
        lane = str(r.get('lane', 'reason'))
        d[lane] = d.get(lane, 0) + 1
    return dict(sorted(d.items(), key=lambda kv: (-kv[1], kv[0])))

# -----------------------------
# 3. Prompt / target builder
# -----------------------------
def build_prompt(row: Dict[str, Any]) -> str:
    return (
        'You are solving a reasoning benchmark problem. '\
        'Give concise reasoning, then put the final answer in exactly one LaTeX boxed expression.\n\n'
        f'Category: {row.get("category", "unknown")}\n'
        f'Problem:\n{row["prompt"]}\n\n'
        'Solution:\n'
    )

def build_target(row: Dict[str, Any]) -> str:
    ans = normalize_answer(row['answer'])
    rationale = sanitize_text(row.get('rationale', ''))
    final = f'The answer is {boxed(ans)}'
    if rationale:
        return rationale.rstrip() + '\n' + final
    return final


# -----------------------------
# 4. Kaggle offline runtime dependency preflight
# -----------------------------
def _has_module(import_name: str) -> bool:
    import importlib.util
    return importlib.util.find_spec(import_name) is not None

def _module_version(import_name: str) -> Optional[str]:
    try:
        from importlib import metadata
        return metadata.version(import_name.replace('_', '-'))
    except Exception:
        try:
            mod = __import__(import_name)
            return str(getattr(mod, '__version__', '0'))
        except Exception:
            return None

def _version_tuple(v: Optional[str]) -> Tuple[int, ...]:
    if not v:
        return tuple()
    nums = re.findall(r'\d+', str(v).split('+')[0])
    return tuple(int(x) for x in nums[:4])

def _version_at_least(found: Optional[str], minimum: str) -> bool:
    f = _version_tuple(found)
    m = _version_tuple(minimum)
    if not f:
        return False
    return f >= m

def _wheel_score(path: Path) -> Tuple[int, str]:
    """Prefer wheels matching current Python, then generic py3/none wheels."""
    import sys
    name = path.name.lower()
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    score = 0
    if py_tag in name:
        score += 100
    if "abi3" in name:
        score += 20
    if "none-any" in name or "py3" in name:
        score += 10
    # Prefer Linux x86_64/manylinux wheels on Kaggle.
    if any(x in name for x in ["linux", "manylinux", "x86_64"]):
        score += 10
    return (-score, name)

def _candidate_package_roots(import_name: str) -> List[Path]:
    roots = []
    base = Path(cfg.input_root)
    if not base.exists():
        return roots
    target_parts = import_name.split('.')
    for p in base.rglob(target_parts[0]):
        try:
            if p.is_dir() and ((p / '__init__.py').exists() or list(p.glob('*.so'))):
                roots.append(p.parent)
        except Exception:
            pass
    return roots

def _candidate_wheels(package_name: str) -> List[Path]:
    base = Path(cfg.input_root)
    if not base.exists():
        return []
    tokens = {package_name.lower().replace('-', '_'), package_name.lower().replace('_', '-')}
    wheels = []
    for p in base.rglob('*.whl'):
        n = p.name.lower()
        if any(t in n for t in tokens):
            wheels.append(p)
    wheels.sort(key=_wheel_score)
    return wheels

def _pip_install_wheel(wheel: Path) -> None:
    import subprocess, sys
    log(f'offline installing wheel: {wheel}')
    cmd = [sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--force-reinstall', str(wheel)]
    subprocess.check_call(cmd)

def ensure_importable(import_name: str, package_name: str, required: bool = True) -> bool:
    """Make a Kaggle offline package importable from attached datasets/wheels.

    This is required for Nemotron-H because the model code imports mamba_ssm before
    the base model can even load. We do not create fake modules. Either a real
    package is present/installed, or training fails loudly.
    """
    if _has_module(import_name):
        log(f'dependency present: {import_name}')
        return True

    import sys
    for root in _candidate_package_roots(import_name):
        if str(root) not in sys.path:
            sys.path.insert(0, str(root))
            log(f'added package root to sys.path for {import_name}: {root}')
        if _has_module(import_name):
            log(f'dependency present via sys.path: {import_name}')
            return True

    errors = []
    for wheel in _candidate_wheels(package_name):
        try:
            _pip_install_wheel(wheel)
            if _has_module(import_name):
                log(f'dependency installed: {import_name}')
                return True
        except Exception as e:
            errors.append(f'{wheel}: {e}')
            log(f'wheel install failed for {wheel}: {e}')

    msg = (
        f'Missing required dependency {import_name!r} for the pinned Nemotron model. '
        f'Attach a Kaggle dataset containing a Python-compatible wheel/package for {package_name!r} '
        f'(and usually causal-conv1d) before running training. No fake adapter will be created.'
    )
    if errors:
        msg += '\nWheel attempts:\n' + '\n'.join(errors[:20])
    if required:
        raise Fatal(msg)
    log('optional dependency unavailable: ' + msg)
    return False

def ensure_importable_min_version(import_name: str, package_name: str, minimum: str, required: bool = True) -> bool:
    """Ensure a binary package exists and meets a minimum version.

    Transformers now rejects old/missing bitsandbytes for 4-bit loading.
    We install a real offline wheel from /kaggle/input when needed.
    """
    if _has_module(import_name):
        found = _module_version(import_name)
        if _version_at_least(found, minimum):
            log(f'dependency present: {import_name}=={found} >= {minimum}')
            return True
        log(f'dependency {import_name} version {found} is below required {minimum}; looking for offline wheel')

    import sys
    for root in _candidate_package_roots(import_name):
        if str(root) not in sys.path:
            sys.path.insert(0, str(root))
            log(f'added package root to sys.path for {import_name}: {root}')
        found = _module_version(import_name)
        if _has_module(import_name) and _version_at_least(found, minimum):
            log(f'dependency present via sys.path: {import_name}=={found}')
            return True

    errors = []
    for wheel in _candidate_wheels(package_name):
        try:
            _pip_install_wheel(wheel)
            found = _module_version(import_name)
            if _has_module(import_name) and _version_at_least(found, minimum):
                log(f'dependency installed: {import_name}=={found}')
                return True
            errors.append(f'{wheel}: installed but version is {found}, need >= {minimum}')
        except Exception as e:
            errors.append(f'{wheel}: {e}')
            log(f'wheel install failed for {wheel}: {e}')

    msg = (
        f'Missing required dependency {import_name!r}>={minimum} for QLoRA 4-bit loading. '
        f'Attach a Kaggle dataset containing a Python 3.12/Linux-compatible wheel for {package_name!r}. '
        f'This notebook will not silently switch to copying an existing adapter or writing dummy tensors.'
    )
    if errors:
        msg += '\nWheel attempts:\n' + '\n'.join(errors[:20])
    if required:
        raise Fatal(msg)
    log('optional dependency unavailable: ' + msg)
    return False

def ensure_nemotron_runtime_deps() -> None:
    """Install/import hard binary deps before AutoModel loads remote Nemotron code."""
    # BitsAndBytes must be present before Transformers builds the 4-bit quantizer.
    if cfg.qlora_4bit:
        ensure_importable_min_version('bitsandbytes', 'bitsandbytes', cfg.bitsandbytes_min_version, required=True)
    # causal_conv1d is often needed by mamba_ssm. Install it first when available.
    ensure_importable('causal_conv1d', 'causal_conv1d', required=False)
    ensure_importable('mamba_ssm', 'mamba_ssm', required=True)

# -----------------------------
# 4. Imports and model loading
# -----------------------------
def import_training_stack():
    ensure_nemotron_runtime_deps()
    import torch
    from torch.utils.data import Dataset
    from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
    try:
        from transformers import BitsAndBytesConfig
    except Exception:
        BitsAndBytesConfig = None
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    return torch, Dataset, AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig, LoraConfig, get_peft_model, prepare_model_for_kbit_training

def load_tokenizer(AutoTokenizer):
    tok = AutoTokenizer.from_pretrained(cfg.base_model_name, trust_remote_code=True, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token or tok.unk_token
    tok.padding_side = 'right'
    return tok

def cuda_inventory(torch) -> Dict[str, Any]:
    info: Dict[str, Any] = {'cuda_available': bool(torch.cuda.is_available()), 'device_count': 0, 'devices': []}
    if not torch.cuda.is_available():
        return info
    info['device_count'] = int(torch.cuda.device_count())
    for i in range(info['device_count']):
        props = torch.cuda.get_device_properties(i)
        total_gib = float(props.total_memory) / (1024 ** 3)
        try:
            free_b, total_b = torch.cuda.mem_get_info(i)
            free_gib = float(free_b) / (1024 ** 3)
        except Exception:
            free_gib = None
        info['devices'].append({
            'index': i,
            'name': props.name,
            'total_gib': round(total_gib, 3),
            'free_gib': None if free_gib is None else round(free_gib, 3),
        })
    return info

def cuda_cleanup(torch) -> None:
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception:
            pass

def _gpu_memory_caps(inv: Dict[str, Any], mode: str) -> Optional[Dict[Any, str]]:
    """Return max_memory caps. None means let Accelerate choose.

    Past working notebooks used device_map="auto" and fp16 on T4. The failure came
    from forcing balanced with almost-full per-GPU caps. This helper gives a retry
    ladder: first no caps, then conservative caps that reserve more on GPU0.
    """
    if mode == 'none':
        return None
    devices = inv.get('devices', [])
    caps: Dict[Any, str] = {'cpu': cfg.cpu_offload_memory}
    for d in devices:
        idx = int(d['index'])
        total = float(d['total_gib'])
        if mode == 'conservative':
            # GPU0 needs extra materialization headroom in Transformers remote-code load.
            cap = 9.0 if idx == 0 else min(12.0, total - 2.5)
        elif mode == 'very_conservative':
            cap = 7.5 if idx == 0 else 10.5
        else:
            cap = min(total - cfg.gpu_headroom_gib, cfg.single_gpu_max_memory_gib)
        cap = max(4.0, cap)
        caps[idx] = f'{cap:.1f}GiB'
    return caps

def loader_attempts(torch) -> List[Tuple[Any, Optional[Dict[Any, str]], str]]:
    inv = cuda_inventory(torch)
    write_json(Path(cfg.work_dir) / 'cuda_inventory.json', inv)
    log('CUDA inventory: ' + json.dumps(inv, ensure_ascii=False))
    n = int(inv.get('device_count', 0))
    if n <= 0:
        raise Fatal('No CUDA GPU is available. Fresh training the 30B adapter requires GPU memory.')

    model_id = cfg.base_model_name.lower()
    looks_30b = any(x in model_id for x in ['30b', 'nano-30b'])
    if looks_30b and n < 2 and cfg.require_two_gpus_for_30b:
        raise Fatal('Only one CUDA GPU is exposed. Use Kaggle GPU T4 x2 or stronger for fresh QLoRA training.')

    forced = cfg.force_device_map.strip() if cfg.force_device_map else 'auto'
    attempts: List[Tuple[Any, Optional[Dict[Any, str]], str]] = []

    # Prior working pattern: fp16 + device_map="auto" + low_cpu_mem_usage.
    # Do this first without max_memory because forced caps caused GPU0 materialization OOM.
    attempts.append((forced or 'auto', None, 'prior_fix_auto_no_caps'))

    # Retry with caps that reserve more GPU0 materialization headroom.
    attempts.append(('auto', _gpu_memory_caps(inv, 'conservative'), 'auto_conservative_gpu0_headroom'))

    # Final retry: sequential placement + heavier CPU offload. Slow, but can finish model materialization.
    attempts.append(('sequential', _gpu_memory_caps(inv, 'very_conservative'), 'sequential_very_conservative'))
    return attempts

def _strict_offload_attempts(torch) -> List[Tuple[Any, Dict[Any, str], str]]:
    """V19 offload lock.

    The previous crash happened before training, while Transformers materialized
    the 30B checkpoint. Uncapped device_map="auto" can temporarily fill GPU0.
    Therefore the first attempt is capped + CPU/disk offload, not uncapped auto.
    """
    inv = cuda_inventory(torch)
    write_json(Path(cfg.work_dir) / 'cuda_inventory.json', inv)
    log('CUDA inventory: ' + json.dumps(inv, ensure_ascii=False))
    if int(inv.get('device_count', 0)) <= 0:
        raise Fatal('No CUDA GPU is available. Fresh training the 30B adapter requires GPU memory.')

    n = int(inv.get('device_count', 0))
    if n >= 2:
        return [
            ('auto', {'cpu': cfg.cpu_load_cap, 0: f'{cfg.gpu0_load_cap_gib:.1f}GiB', 1: f'{cfg.gpu1_load_cap_gib:.1f}GiB'}, 'auto_strict_gpu0_headroom_disk_offload'),
            ('sequential', {'cpu': cfg.cpu_load_cap, 0: '4.5GiB', 1: '10.0GiB'}, 'sequential_deep_offload'),
            ('auto', {'cpu': cfg.cpu_load_cap, 0: '4.0GiB', 1: '8.0GiB'}, 'auto_emergency_cpu_heavy'),
        ]
    return [
        ('auto', {'cpu': cfg.cpu_load_cap, 0: '7.0GiB'}, 'single_gpu_cpu_heavy'),
        ('sequential', {'cpu': cfg.cpu_load_cap, 0: '5.0GiB'}, 'single_gpu_deep_offload'),
    ]

def load_base_model(torch, AutoModelForCausalLM, BitsAndBytesConfig):
    """V19 offload-lock loader.

    Purpose: prevent DeadKernel/OOM during checkpoint materialization by applying
    CPU/disk offload from the first from_pretrained call.
    """
    cuda_cleanup(torch)
    Path(cfg.offload_folder).mkdir(parents=True, exist_ok=True)
    os.makedirs(cfg.offload_folder, exist_ok=True)

    if cfg.qlora_4bit and BitsAndBytesConfig is None:
        raise Fatal('QLORA_4BIT=1 but BitsAndBytesConfig is unavailable. Attach/install bitsandbytes or set QLORA_4BIT=0.')

    inv = cuda_inventory(torch)
    dtype = torch.float16
    if cfg.bf16:
        names = ' '.join(str(d.get('name','')).lower() for d in inv.get('devices', []))
        if 't4' in names:
            log('BF16 requested but T4 detected; forcing FP16.')
        elif torch.cuda.is_available() and torch.cuda.is_bf16_supported():
            dtype = torch.bfloat16

    quant_cfg = None
    if cfg.qlora_4bit and torch.cuda.is_available():
        # NF4 + double quant is the lowest-memory legal training path here.
        quant_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            llm_int8_enable_fp32_cpu_offload=True,
        )

    last_error = None
    for device_map, max_memory, label in _strict_offload_attempts(torch):
        cuda_cleanup(torch)
        kwargs = dict(
            trust_remote_code=True,
            device_map=device_map,
            max_memory=max_memory,
            offload_folder=cfg.offload_folder,
            offload_state_dict=True,
            low_cpu_mem_usage=True,
            torch_dtype=dtype,
            use_safetensors=True,
        )
        if quant_cfg is not None:
            kwargs['quantization_config'] = quant_cfg
        log(f'loading base model with v20_glyph8_r32_offloadlock attempt={label} device_map={device_map!r} max_memory={max_memory} dtype={dtype} qlora_4bit={bool(quant_cfg)} offload_folder={cfg.offload_folder}')
        try:
            model = AutoModelForCausalLM.from_pretrained(cfg.base_model_name, **kwargs)
            try:
                model.config.use_cache = False
            except Exception:
                pass
            try:
                log('hf_device_map=' + json.dumps(getattr(model, 'hf_device_map', {}), default=str)[:4000])
                write_json(Path(cfg.work_dir) / 'hf_device_map.json', getattr(model, 'hf_device_map', {}))
            except Exception:
                pass
            return model
        except ImportError as e:
            msg = str(e).lower()
            if 'bitsandbytes' in msg:
                raise Fatal(f'Need bitsandbytes>={cfg.bitsandbytes_min_version} offline wheel for 4-bit QLoRA. Original error: {e}')
            if 'mamba' in msg or 'causal' in msg:
                raise Fatal(f'Need real mamba_ssm/causal_conv1d offline wheels. Original error: {e}')
            raise
        except BaseException as e:
            last_error = e
            msg = str(e).lower()
            if 'out of memory' in msg or 'cuda' in msg and 'memory' in msg or e.__class__.__name__ in {'OutOfMemoryError'}:
                log(f'load attempt {label} hit memory failure; clearing cache and retrying next stricter placement.')
                cuda_cleanup(torch)
                continue
            raise

    inv = cuda_inventory(torch)
    raise Fatal(
        'OOM/DeadKernel risk persisted after strict CPU/disk offload attempts. '
        'At this point, a T4x2 notebook cannot fresh-train this pinned 30B model under the current container without either: '
        '(1) stronger GPU/RAM, (2) a prebuilt train-time quantized model artifact, or (3) training the adapter outside Kaggle and packaging only adapter_config.json + adapter_model.safetensors. '
        f'CUDA inventory: {json.dumps(inv)}. Last error: {last_error}'
    )

# -----------------------------
# 5. Dynamic LoRA target discovery
# -----------------------------
LORA_CANDIDATES_SCORE = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj',
    'linear_qkv', 'linear_proj', 'linear_fc1', 'linear_fc2',
    'in_proj', 'out_proj', 'x_proj', 'dt_proj',
]

# v18 memory-lock target set. Excludes Mamba conv/dt/x and MLP gate/up/down by default.
# This was the prior working pattern that avoids oversized adapter gradients on Kaggle T4.
LORA_CANDIDATES_V18 = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj', 'in_proj', 'out_proj'
]

LORA_CANDIDATES_MEMORY = [
    'q_proj', 'v_proj', 'o_proj'
]

def active_lora_candidates() -> List[str]:
    override = [x.strip() for x in str(cfg.lora_target_modules).split(',') if x.strip()]
    if override:
        return override
    if cfg.lora_target_profile == 'minimal':
        return LORA_CANDIDATES_MEMORY
    if cfg.lora_target_profile == 'score':
        return LORA_CANDIDATES_SCORE
    return LORA_CANDIDATES_V18

def discover_lora_targets(model) -> List[str]:
    present = set()
    for name, module in model.named_modules():
        leaf = name.split('.')[-1]
        candidates = active_lora_candidates()
        if leaf not in candidates:
            continue
        w = getattr(module, 'weight', None)
        if w is None:
            continue
        try:
            if len(tuple(w.shape)) == 2:
                present.add(leaf)
        except Exception:
            present.add(leaf)
    targets = [x for x in active_lora_candidates() if x in present]
    if not targets:
        raise Fatal('No valid LoRA target modules discovered. Refusing to train a no-op adapter.')
    log(f'LoRA targets discovered: {targets}')
    write_json(Path(cfg.work_dir) / 'lora_targets.json', {'targets': targets})
    return targets

# -----------------------------
# 6. Encoding + score-aware weighted dataset
# -----------------------------
def encode_row(row: Dict[str, Any], tokenizer) -> Dict[str, Any]:
    prompt = build_prompt(row)
    target = build_target(row)
    eos = tokenizer.eos_token or ''

    prompt_ids = tokenizer.encode(prompt, add_special_tokens=True)
    ans = normalize_answer(row['answer'])
    final_line = f'The answer is {boxed(ans)}'

    # Split target into segments so final boxed answer receives higher loss.
    if target.endswith(final_line):
        pre = target[:-len(final_line)]
        final = final_line
    else:
        pre = ''
        final = target
    box_text = boxed(ans)
    box_at = final.find(box_text)
    if box_at >= 0:
        final_pre = final[:box_at]
        final_box = box_text
        final_post = final[box_at + len(box_text):]
    else:
        final_pre, final_box, final_post = final, '', ''

    segs = []
    if pre:
        segs.append((pre, cfg.reasoning_loss_weight))
    if final_pre:
        segs.append((final_pre, cfg.final_line_loss_weight))
    if final_box:
        segs.append((final_box, cfg.boxed_answer_loss_weight))
    if final_post:
        segs.append((final_post, cfg.final_line_loss_weight))
    if eos:
        segs.append((eos, cfg.final_line_loss_weight))

    target_ids: List[int] = []
    target_weights: List[float] = []
    for text, wt in segs:
        ids = tokenizer.encode(text, add_special_tokens=False)
        target_ids.extend(ids)
        target_weights.extend([float(wt) * float(row.get('score_weight', 1.0))] * len(ids))

    # Trim from the left of prompt first, preserving target/final answer.
    max_target = max(1, cfg.max_seq_len - 16)
    if len(target_ids) > max_target:
        target_ids = target_ids[-max_target:]
        target_weights = target_weights[-max_target:]
    room = cfg.max_seq_len - len(target_ids)
    if room < 8:
        room = 8
        target_ids = target_ids[-(cfg.max_seq_len-room):]
        target_weights = target_weights[-(cfg.max_seq_len-room):]
    prompt_ids = prompt_ids[-room:]

    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids
    loss_weights = [0.0] * len(prompt_ids) + target_weights
    attention_mask = [1] * len(input_ids)
    return {
        'input_ids': input_ids,
        'labels': labels,
        'attention_mask': attention_mask,
        'loss_weights': loss_weights,
        'category': row.get('category', 'unknown'),
        'id': row.get('id'),
    }

class ScoreAwareDataset:
    def __init__(self, rows: List[Dict[str, Any]], tokenizer):
        self.rows = rows
        self.tokenizer = tokenizer
        self.cache: Dict[int, Dict[str, Any]] = {}
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx: int):
        if idx not in self.cache:
            self.cache[idx] = encode_row(self.rows[idx], self.tokenizer)
        return self.cache[idx]

class ScoreAwareCollator:
    def __init__(self, tokenizer):
        self.pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        import torch
        max_len = max(len(f['input_ids']) for f in features)
        batch = {'input_ids': [], 'attention_mask': [], 'labels': [], 'loss_weights': []}
        for f in features:
            n = max_len - len(f['input_ids'])
            batch['input_ids'].append(f['input_ids'] + [self.pad_id] * n)
            batch['attention_mask'].append(f['attention_mask'] + [0] * n)
            batch['labels'].append(f['labels'] + [-100] * n)
            batch['loss_weights'].append(f['loss_weights'] + [0.0] * n)
        return {k: torch.tensor(v, dtype=torch.long if k != 'loss_weights' else torch.float32) for k, v in batch.items()}

def make_weighted_trainer_class(Trainer):
    import torch
    import torch.nn.functional as F
    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop('labels')
            loss_weights = inputs.pop('loss_weights')
            outputs = model(**inputs)
            logits = outputs.logits
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            shift_weights = loss_weights[:, 1:].contiguous()
            vocab = shift_logits.size(-1)
            flat_logits = shift_logits.view(-1, vocab)
            flat_labels = shift_labels.view(-1)
            flat_weights = shift_weights.view(-1)
            mask = flat_labels.ne(-100)
            safe_labels = flat_labels.masked_fill(~mask, 0)
            token_loss = F.cross_entropy(flat_logits, safe_labels, reduction='none')
            weighted = token_loss * flat_weights * mask.float()
            denom = (flat_weights * mask.float()).sum().clamp_min(1.0)
            loss = weighted.sum() / denom
            return (loss, outputs) if return_outputs else loss
    return WeightedTrainer

# -----------------------------
# 7. Base-model mapping before steering
# -----------------------------
def base_loss_for_encoded(model, torch, encoded: Dict[str, Any], device=None) -> float:
    labels = torch.tensor([encoded['labels']], dtype=torch.long, device=device or model.device)
    input_ids = torch.tensor([encoded['input_ids']], dtype=torch.long, device=device or model.device)
    attention_mask = torch.tensor([encoded['attention_mask']], dtype=torch.long, device=device or model.device)
    with torch.no_grad():
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = out.logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        mask = shift_labels.ne(-100)
        safe = shift_labels.masked_fill(~mask, 0)
        import torch.nn.functional as F
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), safe.view(-1), reduction='none')
        loss = loss.view_as(safe)
        denom = mask.float().sum().clamp_min(1.0)
        return float((loss * mask.float()).sum().detach().cpu() / denom.detach().cpu())

def run_mapping_probe(rows: List[Dict[str, Any]], tokenizer, model=None, torch=None) -> List[Dict[str, Any]]:
    # Always write category map first.
    summary = summarize_rows(rows)
    log(f'row category counts: {summary["category_counts"]}')
    if not cfg.probe_base_loss or model is None or torch is None:
        write_json(Path(cfg.work_dir) / 'mapping_summary.json', summary)
        return rows

    # Probe a representative subset across categories.
    by_cat: Dict[str, List[int]] = {}
    for i, r in enumerate(rows):
        by_cat.setdefault(r['category'], []).append(i)
    probe_indices = []
    per_cat = max(1, cfg.probe_max_rows // max(1, len(by_cat)))
    for cat in CATEGORY_ORDER + sorted(set(by_cat) - set(CATEGORY_ORDER)):
        ids = by_cat.get(cat, [])
        if ids:
            probe_indices.extend(ids[:per_cat])
    probe_indices = probe_indices[:cfg.probe_max_rows]
    log(f'probing base-model target loss on {len(probe_indices)} rows before LoRA training')

    losses = []
    for n, idx in enumerate(probe_indices, 1):
        enc = encode_row(rows[idx], tokenizer)
        try:
            loss = base_loss_for_encoded(model, torch, enc)
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                torch.cuda.empty_cache()
            log(f'probe skipped row {idx}: {e}')
            continue
        rows[idx]['base_loss'] = loss
        losses.append(loss)
        if n % 16 == 0:
            log(f'probe {n}/{len(probe_indices)} avg_loss={sum(losses)/len(losses):.4f}')

    if losses:
        lo, hi = min(losses), max(losses)
        span = max(1e-6, hi - lo)
        # Higher base loss = model finds target harder = slightly more steering.
        for r in rows:
            if r.get('base_loss') is not None:
                hard = (float(r['base_loss']) - lo) / span
                r['score_weight'] = float(r.get('score_weight', 1.0)) * (1.0 + 0.75 * hard)
        probe_summary = {
            **summary,
            'probe_rows': len(losses),
            'base_loss_min': lo,
            'base_loss_max': hi,
            'base_loss_mean': sum(losses) / len(losses),
        }
    else:
        probe_summary = {**summary, 'probe_rows': 0}
    write_json(Path(cfg.work_dir) / 'mapping_summary.json', probe_summary)
    return rows


# -----------------------------
# 8A. Model Weight Atlas v2 (competition-safe index + PEFT steering map)
# -----------------------------
# Corrected MWA for Kaggle/PEFT:
# - Atlas holds coordinates and steering signals only.
# - No clone of W0, no SVD of W0, no residual bank.
# - LoRA weights are created by official PEFT so adapter_config.json and
#   adapter_model.safetensors are Kaggle-valid.
# - The Atlas is saved as a work/report JSON only; it is never placed in submission.zip.

GLYPH_IDS = [f"G{i}" for i in range(16)]
MWA_DOMAINS = [
    'bit_manipulation',
    'substitution_cipher',
    'numeral_conversion',
    'unit_conversion',
    'equation_numeric',
    'physics_gravity',
    'unknown',
]

class ModelWeightAtlasV2:
    """
    Zero-copy, Google-Maps-style model coordinate atlas.

    It is an index, not a weight bank. It stores WHERE LoRA can attach and
    WHY the score-aware curriculum wants steering there. Actual BA tensors are
    created and trained by PEFT.
    """

    def __init__(self, rows_summary: Dict[str, Any]):
        self.rows_summary = rows_summary
        self.records: List[Dict[str, Any]] = []
        self.layer_count: int = 0
        self.module_counts: Dict[str, int] = {}
        self.steering_map: Dict[str, Any] = {}
        self.summary: Dict[str, Any] = {
            'kind': 'ModelWeightAtlasV2',
            'version': 'v2_index_only_peft_writer_v18_memorylock',
            'glyph_count': len(GLYPH_IDS),
            'rank': cfg.lora_rank,
            'alpha': cfg.lora_alpha,
            'dropout': cfg.lora_dropout,
            'lora_math': 'h = W0*x + (alpha/r)*B*A*x; W0 frozen; BA learned as low-rank delta W',
            'init_policy': 'PEFT LoRA default: A random, B zero, adapter contribution starts near zero',
            'submission_policy': 'submission.zip root contains exactly adapter_config.json and adapter_model.safetensors',
            'not_in_submission': ['atlas json', 'tokenizer files', 'base model files', 'residual banks', 'full weights'],
            'warning': 'metadata only; no full-weight clone, no SVD of W0, no manual PEFT state dict fabrication',
        }

    @staticmethod
    def _extract_layer(name: str) -> int:
        # Handles common HF names: model.layers.12, layers.12, h.12, blocks.12
        m = re.search(r'(?:^|\.)(?:layers|h|blocks)\.(\d+)(?:\.|$)', name)
        return int(m.group(1)) if m else -1

    @staticmethod
    def _domain_for(layer: int, module: str, row_cats: Dict[str, int]) -> str:
        # Keep deterministic but score-aware. Early/mid/late preference is only
        # for atlas annotation; PEFT target_modules still controls real injection.
        if module in {'q_proj', 'k_proj', 'v_proj', 'linear_qkv', 'in_proj'}:
            preferred = ['bit_manipulation', 'substitution_cipher', 'numeral_conversion']
        elif module in {'o_proj', 'linear_proj', 'out_proj'}:
            preferred = ['unit_conversion', 'equation_numeric', 'physics_gravity']
        else:
            preferred = ['equation_numeric', 'physics_gravity', 'unknown']
        # Pick the highest-count available preferred domain, fallback to largest cat.
        available = [(row_cats.get(d, 0), d) for d in preferred]
        if max(x[0] for x in available) > 0:
            return max(available)[1]
        if row_cats:
            return max((v, k) for k, v in row_cats.items())[1]
        return 'unknown'

    @staticmethod
    def _intensity(layer: int, layer_count: int, domain: str) -> float:
        if layer < 0 or layer_count <= 1:
            return 0.5
        # Bell-curve centers by problem family. This is a steering heatmap, not a
        # hard layer mask, unless PEFT layers_to_transform is deliberately enabled later.
        centers = {
            'bit_manipulation': 0.25,
            'substitution_cipher': 0.30,
            'numeral_conversion': 0.35,
            'unit_conversion': 0.55,
            'equation_numeric': 0.68,
            'physics_gravity': 0.72,
            'unknown': 0.50,
        }
        c = centers.get(domain, 0.50) * max(1, layer_count - 1)
        sigma = max(2.0, layer_count / 6.0)
        return float(math.exp(-0.5 * ((layer - c) / sigma) ** 2))

    @classmethod
    def from_model(cls, model, candidates: Sequence[str], rows_summary: Dict[str, Any]) -> 'ModelWeightAtlasV2':
        atlas = cls(rows_summary)
        candidates = list(candidates)
        row_cats = dict(rows_summary.get('category_counts') or {})
        idx = 0
        max_layer = -1

        for name, module in model.named_modules():
            leaf = name.split('.')[-1]
            if leaf not in candidates:
                continue
            w = getattr(module, 'weight', None)
            if w is None:
                continue
            try:
                shape = tuple(int(x) for x in w.shape)
            except Exception:
                shape = ()
            if len(shape) != 2:
                continue

            layer = cls._extract_layer(name)
            max_layer = max(max_layer, layer)
            glyph = GLYPH_IDS[idx % len(GLYPH_IDS)]
            tile_x = idx % 4
            tile_y = (idx // 4) % 4
            domain = cls._domain_for(layer, leaf, row_cats)

            atlas.records.append({
                'coordinate': f'L{layer}:{leaf}:{tile_x},{tile_y}:{glyph}:0-{cfg.lora_rank}',
                'layer': layer,
                'module': leaf,
                'module_path': name,
                'shape': list(shape),
                'glyph_id': glyph,
                'tile_x': tile_x,
                'tile_y': tile_y,
                'target_domain': domain,
                'rank': cfg.lora_rank,
                'alpha': cfg.lora_alpha,
                'dropout': cfg.lora_dropout,
                'base_dtype': str(getattr(w, 'dtype', 'unknown')),
                'base_device': str(getattr(w, 'device', 'unknown')),
                'is_weight_bank': False,
            })
            atlas.module_counts[leaf] = atlas.module_counts.get(leaf, 0) + 1
            idx += 1

        atlas.layer_count = max_layer + 1 if max_layer >= 0 else 0
        # Add intensity after layer_count is known.
        for rec in atlas.records:
            rec['steering_intensity'] = cls._intensity(int(rec['layer']), atlas.layer_count, str(rec['target_domain']))

        # Domain heatmap: small JSON only.
        for domain in MWA_DOMAINS:
            intensities = [cls._intensity(i, atlas.layer_count, domain) for i in range(atlas.layer_count)]
            preferred = sorted({r['glyph_id'] for r in atlas.records if r.get('target_domain') == domain})
            atlas.steering_map[domain] = {
                'layer_intensities': intensities,
                'preferred_glyphs': preferred,
                'row_count': row_cats.get(domain, 0),
                'category_weight': CATEGORY_WEIGHTS.get(domain, CATEGORY_WEIGHTS.get('unknown', 1.0)),
            }

        atlas.summary.update({
            'layer_count': atlas.layer_count,
            'coordinate_count': len(atlas.records),
            'candidate_modules': candidates,
            'module_counts': atlas.module_counts,
            'row_map': rows_summary,
            'steering_map_domains': list(atlas.steering_map),
        })
        return atlas

    def attach_lora_targets(self, targets: Sequence[str]) -> None:
        targets = set(targets)
        for rec in self.records:
            rec['lora_targeted_by_peft'] = bool(rec.get('module') in targets)
        self.summary['peft_target_modules'] = sorted(targets)
        self.summary['targeted_coordinate_count'] = sum(1 for r in self.records if r.get('lora_targeted_by_peft'))
        self.summary['targeted_module_counts'] = {
            m: sum(1 for r in self.records if r.get('module') == m and r.get('lora_targeted_by_peft'))
            for m in sorted(targets)
        }

    def collect_lora_stats(self, model, torch_module) -> None:
        # Stats only. Does not alter tensors. Useful for verifying non-empty trained adapter.
        stats = []
        for name, p in model.named_parameters():
            if 'lora_A' not in name and 'lora_B' not in name:
                continue
            with torch_module.no_grad():
                d = p.detach()
                try:
                    f = d.float()
                    norm = float(f.norm().cpu().item())
                    mean_abs = float(f.abs().mean().cpu().item())
                    nonzero = int((f != 0).sum().cpu().item())
                except Exception:
                    norm, mean_abs, nonzero = None, None, None
            stats.append({
                'name': name,
                'shape': [int(x) for x in tuple(p.shape)],
                'dtype': str(p.dtype),
                'device': str(p.device),
                'norm': norm,
                'mean_abs': mean_abs,
                'nonzero': nonzero,
            })
        self.summary['lora_tensor_count'] = len(stats)
        self.summary['lora_tensor_norm_sum'] = sum(float(x.get('norm') or 0.0) for x in stats)
        self.summary['lora_tensors_preview'] = stats[:200]

    def save(self, path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            'summary': self.summary,
            'steering_map': self.steering_map,
            'coordinates': self.records,
        }
        path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')
        log(f'MWA-v2 index saved outside submission: {path} coordinates={len(self.records)}')

def build_mwa_rows_summary(rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    cat: Dict[str, int] = {}
    src: Dict[str, int] = {}
    for r in rows:
        c = r.get('category', 'unknown')
        s = r.get('source', 'unknown')
        cat[c] = cat.get(c, 0) + 1
        src[s] = src.get(s, 0) + 1
    return {
        'rows': len(rows),
        'category_counts': cat,
        'source_counts_top': dict(sorted(src.items(), key=lambda kv: -kv[1])[:10]),
        'curriculum_order': CATEGORY_ORDER,
        'category_weights': CATEGORY_WEIGHTS,
        'note': 'Atlas maps curriculum pressure before LoRA steering; official PEFT trains BA tensors.',
    }

# -----------------------------
# 8. Fresh LoRA training
# -----------------------------
def train_fresh_lora(rows: List[Dict[str, Any]]) -> Path:
    torch, Dataset, AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig, LoraConfig, get_peft_model, prepare_model_for_kbit_training = import_training_stack()

    if Path(cfg.adapter_dir).exists():
        shutil.rmtree(cfg.adapter_dir)
    Path(cfg.adapter_dir).mkdir(parents=True, exist_ok=True)

    tokenizer = load_tokenizer(AutoTokenizer)
    model = load_base_model(torch, AutoModelForCausalLM, BitsAndBytesConfig)

    atlas = None
    if cfg.mwa_enabled:
        atlas = ModelWeightAtlasV2.from_model(model, active_lora_candidates(), build_mwa_rows_summary(rows))
        atlas.save(Path(cfg.mwa_output_json))

    # Map before steering.
    rows = run_mapping_probe(rows, tokenizer, model=model, torch=torch)
    if cfg.run_mode == 'map_only':
        log('RUN_MODE=map_only complete. No adapter/submission created.')
        return Path(cfg.adapter_dir)

    if cfg.gradient_checkpointing:
        try:
            model.gradient_checkpointing_enable()
        except Exception:
            pass
        try:
            model.enable_input_require_grads()
        except Exception:
            pass
    try:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=bool(cfg.gradient_checkpointing))
    except TypeError:
        model = prepare_model_for_kbit_training(model)

    targets = discover_lora_targets(model)
    if atlas is not None:
        atlas.attach_lora_targets(targets)
        atlas.save(Path(cfg.mwa_output_json))
    lora_cfg = LoraConfig(
        r=cfg.lora_rank,
        lora_alpha=cfg.lora_alpha,
        target_modules=targets,
        lora_dropout=cfg.lora_dropout,
        bias='none',
        task_type='CAUSAL_LM',
    )
    model = get_peft_model(model, lora_cfg)
    try:
        model.print_trainable_parameters()
    except Exception:
        pass

    # Re-sort by category, then by score_weight descending inside category.
    rows.sort(key=lambda r: (CATEGORY_RANK.get(r['category'], 999), -float(r.get('score_weight', 1.0)), len(r['prompt'])))
    write_json(Path(cfg.work_dir) / 'mapped_training_rows_preview.json', rows[:250])

    train_rows = rows
    WeightedTrainer = make_weighted_trainer_class(Trainer)
    optim = 'paged_adamw_8bit' if cfg.qlora_4bit else 'adamw_torch'

    def _training_args_for_chunk(chunk_idx: int, steps_this_chunk: int):
        args_kwargs = dict(
            output_dir=str(Path(cfg.work_dir) / 'trainer_chunks' / f'chunk_{chunk_idx:04d}'),
            per_device_train_batch_size=cfg.train_batch_size,
            gradient_accumulation_steps=cfg.grad_accum_steps,
            learning_rate=cfg.learning_rate,
            warmup_ratio=cfg.warmup_ratio,
            weight_decay=cfg.weight_decay,
            max_grad_norm=cfg.max_grad_norm,
            logging_steps=1,
            save_strategy='no',
            report_to=[],
            remove_unused_columns=False,
            bf16=bool(cfg.bf16),
            fp16=not bool(cfg.bf16),
            optim=optim,
            gradient_checkpointing=bool(cfg.gradient_checkpointing),
        )
        if steps_this_chunk and steps_this_chunk > 0:
            args_kwargs['max_steps'] = int(steps_this_chunk)
        else:
            args_kwargs['num_train_epochs'] = cfg.num_epochs
        return TrainingArguments(**args_kwargs)

    def _clear_chunk_memory(*objs):
        for obj in objs:
            try:
                del obj
            except Exception:
                pass
        gc.collect()
        if cfg.clear_cache_each_chunk:
            try:
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
            except Exception:
                pass

    if cfg.chunked_training:
        chunk_size = max(1, int(cfg.train_chunk_size))
        total_rows = len(train_rows)
        planned_chunks = math.ceil(total_rows / chunk_size)
        if cfg.max_steps and cfg.max_steps > 0:
            planned_chunks = min(planned_chunks, max(1, math.ceil(cfg.max_steps / max(1, cfg.chunk_max_steps))))
        if cfg.max_train_chunks and cfg.max_train_chunks > 0:
            planned_chunks = min(planned_chunks, cfg.max_train_chunks)
        remaining_steps = int(cfg.max_steps) if cfg.max_steps and cfg.max_steps > 0 else 0
        log(f'ChunkTrainLock enabled: total_rows={total_rows}, chunk_size={chunk_size}, planned_chunks={planned_chunks}, global_max_steps={cfg.max_steps}, chunk_max_steps={cfg.chunk_max_steps}')
        chunk_manifest = []
        for chunk_idx in range(planned_chunks):
            start = chunk_idx * chunk_size
            end = min(start + chunk_size, total_rows)
            if start >= end:
                break
            chunk_rows = train_rows[start:end]
            if remaining_steps > 0:
                steps_this_chunk = min(max(1, int(cfg.chunk_max_steps)), remaining_steps)
                remaining_steps -= steps_this_chunk
            else:
                steps_this_chunk = 0
            log(f'ChunkTrainLock chunk {chunk_idx+1}/{planned_chunks}: rows={start}:{end} n={len(chunk_rows)} steps={steps_this_chunk or "epoch"}')
            train_ds = ScoreAwareDataset(chunk_rows, tokenizer)
            collator = ScoreAwareCollator(tokenizer)
            training_args = _training_args_for_chunk(chunk_idx, steps_this_chunk)
            trainer = WeightedTrainer(model=model, args=training_args, train_dataset=train_ds, data_collator=collator)
            trainer.train()
            chunk_meta = {'chunk': chunk_idx, 'row_start': start, 'row_end': end, 'rows': len(chunk_rows), 'steps': steps_this_chunk}
            chunk_manifest.append(chunk_meta)
            write_json(Path(cfg.work_dir) / 'chunk_train_manifest.json', chunk_manifest)
            if cfg.save_after_each_chunk:
                model.save_pretrained(cfg.adapter_dir, safe_serialization=True)
                log(f'ChunkTrainLock saved adapter checkpoint after chunk {chunk_idx+1}: {cfg.adapter_dir}')
            _clear_chunk_memory(trainer, training_args, train_ds, collator)
            if remaining_steps == 0 and cfg.max_steps and cfg.max_steps > 0:
                log('ChunkTrainLock reached global MAX_STEPS; stopping chunk loop.')
                break
    else:
        train_ds = ScoreAwareDataset(train_rows, tokenizer)
        collator = ScoreAwareCollator(tokenizer)
        training_args = _training_args_for_chunk(0, int(cfg.max_steps) if cfg.max_steps else 0)
        trainer = WeightedTrainer(model=model, args=training_args, train_dataset=train_ds, data_collator=collator)
        trainer.train()
        _clear_chunk_memory(trainer, training_args, train_ds, collator)

    if atlas is not None:
        atlas.collect_lora_stats(model, torch)
        atlas.save(Path(cfg.mwa_output_json))

    # Save only the LoRA adapter. Do not save tokenizer into the adapter dir for submission.
    model.save_pretrained(cfg.adapter_dir, safe_serialization=True)

    # Normalize adapter_config base model pointer and enforce rank.
    cfg_path = Path(cfg.adapter_dir) / 'adapter_config.json'
    if not cfg_path.exists():
        raise Fatal('Training finished but adapter_config.json was not written.')
    acfg = json.loads(cfg_path.read_text())
    acfg['base_model_name_or_path'] = cfg.public_base_model_name
    if int(acfg.get('r', cfg.lora_rank)) > MAX_LORA_RANK:
        raise Fatal(f'adapter_config rank r={acfg.get("r")} exceeds {MAX_LORA_RANK}')
    cfg_path.write_text(json.dumps(acfg, indent=2, ensure_ascii=False), encoding='utf-8')

    del trainer
    gc.collect()
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass
    return Path(cfg.adapter_dir)



def inspect_safetensors_header(path: Path) -> Tuple[List[str], Dict[str, Tuple[int, ...]]]:
    """Small safetensors header reader used as a fallback when safetensors is unavailable."""
    import struct
    with path.open('rb') as f:
        raw = f.read(8)
        if len(raw) != 8:
            raise Fatal('safetensors file is too short to contain a header length')
        n = struct.unpack('<Q', raw)[0]
        if n <= 0 or n > 256_000_000:
            raise Fatal(f'invalid safetensors header length: {n}')
        header = f.read(n)
    meta = json.loads(header.decode('utf-8').strip())
    keys = [k for k in meta.keys() if k != '__metadata__']
    shapes = {k: tuple(meta[k].get('shape', [])) for k in keys if isinstance(meta.get(k), dict)}
    return keys, shapes

def write_minimal_safetensors_for_selftest(path: Path) -> None:
    """Write a tiny syntactically valid safetensors file for package-policy selftest only."""
    import struct
    key = 'base_model.model.layers.0.self_attn.q_proj.lora_A.weight'
    nbytes = 32 * 16 * 4
    header_obj = {key: {'dtype': 'F32', 'shape': [32, 16], 'data_offsets': [0, nbytes]}}
    header = json.dumps(header_obj, separators=(',', ':')).encode('utf-8')
    # Padding spaces are accepted because header JSON is parsed after strip().
    pad = (8 - (len(header) % 8)) % 8
    header += b' ' * pad
    path.write_bytes(struct.pack('<Q', len(header)) + header + (b'\x00' * nbytes))

# -----------------------------
# 9. Strict adapter validation and zip packaging
# -----------------------------
def validate_adapter_files(adapter_dir: Path) -> Dict[str, Any]:
    cfg_path = adapter_dir / 'adapter_config.json'
    st_path = adapter_dir / 'adapter_model.safetensors'
    if not cfg_path.exists():
        raise Fatal(f'Missing {cfg_path}')
    if not st_path.exists():
        raise Fatal(f'Missing {st_path}')
    if st_path.stat().st_size < cfg.min_adapter_bytes and cfg.run_mode != 'selftest':
        raise Fatal(f'adapter_model.safetensors too small ({st_path.stat().st_size} bytes). Refusing likely empty adapter.')

    acfg = json.loads(cfg_path.read_text())
    r = int(acfg.get('r', -1))
    if not (1 <= r <= MAX_LORA_RANK):
        raise Fatal(f'Invalid LoRA rank in adapter_config.json: r={r}')

    try:
        from safetensors import safe_open
        keys = []
        shapes = {}
        with safe_open(str(st_path), framework='pt', device='cpu') as f:
            keys = list(f.keys())
            for k in keys[:1000]:
                shapes[k] = tuple(f.get_tensor(k).shape)
    except ModuleNotFoundError:
        keys, shapes = inspect_safetensors_header(st_path)
    except Fatal:
        raise
    except Exception as e:
        raise Fatal(f'Could not inspect adapter_model.safetensors: {e}')

    if not keys:
        raise Fatal('adapter_model.safetensors contains zero tensors.')
    lora_keys = [k for k in keys if 'lora_' in k.lower()]
    if not lora_keys and cfg.run_mode != 'selftest':
        raise Fatal('safetensors has tensors, but no LoRA tensor keys were found.')

    meta = {
        'adapter_dir': str(adapter_dir),
        'adapter_config': str(cfg_path),
        'adapter_model': str(st_path),
        'rank': r,
        'safetensors_bytes': st_path.stat().st_size,
        'tensor_count': len(keys),
        'lora_tensor_count': len([k for k in keys if 'lora_' in k.lower()]),
        'sha256_adapter_model': sha256_file(st_path),
        'sample_shapes': {k: list(v) for k, v in list(shapes.items())[:20]},
    }
    write_json(Path(cfg.work_dir) / 'adapter_validation.json', meta)
    return meta

def create_submission_zip(adapter_dir: Path) -> Path:
    meta = validate_adapter_files(adapter_dir)
    out = Path(cfg.output_zip)
    if out.exists():
        out.unlink()
    with zipfile.ZipFile(out, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        z.write(adapter_dir / 'adapter_config.json', arcname='adapter_config.json')
        z.write(adapter_dir / 'adapter_model.safetensors', arcname='adapter_model.safetensors')
    with zipfile.ZipFile(out, 'r') as z:
        names = sorted(z.namelist())
    expected = ['adapter_config.json', 'adapter_model.safetensors']
    if cfg.exact_zip_two_files and names != expected:
        raise Fatal(f'submission.zip must contain exactly {expected}, got {names}')
    zip_meta = {
        **meta,
        'submission_zip': str(out),
        'submission_zip_bytes': out.stat().st_size,
        'zip_names': names,
        'sha256_submission_zip': sha256_file(out),
    }
    write_json(Path(cfg.work_dir) / 'submission_zip_validation.json', zip_meta)
    log(f'WROTE STRICT SUBMISSION ZIP: {out}')
    log(f'ZIP CONTENTS: {names}')
    return out

# -----------------------------
# 10. Self-test for mapping/package policy only
# -----------------------------
def selftest() -> None:
    import tempfile
    tmp = Path(tempfile.mkdtemp(prefix='fresh_lora_selftest_'))
    ad = tmp / 'adapter'
    ad.mkdir(parents=True)
    (ad / 'adapter_config.json').write_text(json.dumps({
        'r': 32,
        'lora_alpha': 64,
        'peft_type': 'LORA',
        'task_type': 'CAUSAL_LM',
        'base_model_name_or_path': PINNED_BASE_MODEL_PATH,
        'target_modules': ['q_proj'],
    }, indent=2), encoding='utf-8')
    write_minimal_safetensors_for_selftest(ad / 'adapter_model.safetensors')
    old_min = cfg.min_adapter_bytes
    old_mode = cfg.run_mode
    cfg.min_adapter_bytes = 1
    cfg.run_mode = 'selftest'
    create_submission_zip(ad)
    with zipfile.ZipFile(cfg.output_zip) as z:
        assert sorted(z.namelist()) == ['adapter_config.json', 'adapter_model.safetensors']
    cfg.min_adapter_bytes = old_min
    cfg.run_mode = old_mode
    log('selftest passed')

# -----------------------------
# 11. Main
# -----------------------------
def main():
    if cfg.run_mode == 'selftest':
        selftest()
        return

    rows = discover_real_training_rows()
    base_n = len(rows)
    glyph_rows = make_glyph8_records(rows)
    if glyph_rows:
        rows = rows + glyph_rows
        rows.sort(key=lambda r: (CATEGORY_RANK.get(r['category'], 999), str(r.get('lane','reason')), len(r['prompt'])))
        log(f'added {len(glyph_rows)} GLYPH8 transport rows; total rows={len(rows)} base_rows={base_n}')
    summary = summarize_rows(rows)
    log(f'discovered {len(rows)} real rows')
    print(json.dumps(summary, indent=2)[:4000], flush=True)

    adapter_dir = train_fresh_lora(rows)
    if cfg.run_mode == 'map_only':
        return
    create_submission_zip(adapter_dir)
    log('DONE: GLYPH8-R32 V21 ChunkTrainLock fresh mapped LoRA adapter trained and packaged with exactly two root files.')

if __name__ == '__main__':
    main()
